In [29]:
# Import packages
import pandas as pd
import numpy as np

In [30]:
# Load dataset
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "dirty_cafe_sales.csv"

# Load the latest version
sales_df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

/tmp/ipykernel_1440/1350100498.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  sales_df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


Let's first get a feel for what our dataset looks like, and the amount of rows/columns.

In [31]:
sales_df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [32]:
# Print number of rows/columns of df
sales_df.shape

(10000, 8)

Now, let's look for duplicate rows and null values in the dataset.

In [33]:
# Count duplicate rows, amount of nulls in each column
duplicate_rows = sales_df.duplicated().sum()
print("Duplicates rows:", duplicate_rows)

null_counts = sales_df.isnull().sum()
print(null_counts)

Duplicates rows: 0
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


Now, let's dive deeper into each column to check data types and possible incorrect values. It should be noted that null values are classified as the "float" data type.

In [34]:
# Check data type percentages for each column
dtype_percentages = pd.DataFrame({
    col: sales_df[col].apply(lambda x: type(x).__name__).value_counts(normalize=True) * 100
    for col in sales_df.columns
}).T.fillna(0)

dtype_percentages

,float,str
Transaction ID,0.00,100.00
Item,3.33,96.67
Quantity,1.38,98.62
Price Per Unit,1.79,98.21
Total Spent,1.73,98.27
Payment Method,25.79,74.21
Location,32.65,67.35
Transaction Date,1.59,98.41


In [35]:
# Check unique values of columns
item_unique = sales_df['Item'].unique()
quantity_unique = sales_df['Quantity'].unique()
price_unit_unique = sales_df['Price Per Unit'].unique()
total_spent_unique = sales_df['Total Spent'].unique()
payment_unique = sales_df['Payment Method'].unique()
location_unique = sales_df['Location'].unique()
date_unique = sales_df['Transaction Date'].unique()


print("Unique values in Item:", item_unique)
print("Unique values in Quantity:", quantity_unique)
print("Unique values in Price Per Unit:", price_unit_unique)
print("Unique values in Total Spent:", total_spent_unique)
print("Unique values in Payment Method:", payment_unique)
print("Unique values in Location:", location_unique)


Unique values in Item: ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']
Unique values in Quantity: ['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]
Unique values in Price Per Unit: ['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']
Unique values in Total Spent: ['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']
Unique values in Payment Method: ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]
Unique values in Location: ['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']


Two big problems that are already apparent are that we have numeric values that are represented as strings. Additionally, in each column that are a lot of "UNKNOWN" or "ERROR" values. Let's fix both of those problems.

In [36]:
# For each column change "UNKNOWN" and "ERROR" to nulls
for col in sales_df.columns:
    sales_df[col] = sales_df[col].replace(['UNKNOWN', 'ERROR'], np.nan)

null_counts = sales_df.isnull().sum()
print(null_counts)

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [37]:
# Change Quantity, Price Per Unit, and Total Spent to float
sales_df['Quantity'] = sales_df['Quantity'].astype(float)
sales_df['Price Per Unit'] = sales_df['Price Per Unit'].astype(float)
sales_df['Total Spent'] = sales_df['Total Spent'].astype(float)

Let's check for outliers in our numeric columns.

In [38]:
# Identify major outliers in our numeric columns
numeric_cols = ['Price Per Unit', 'Quantity', 'Total Spent']

outlier_counts = {}

for col in numeric_cols:
    Q1 = sales_df[col].quantile(0.25)
    Q3 = sales_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR

    outliers = (sales_df[col] < lower_bound) | (sales_df[col] > upper_bound)

    outlier_counts[col] = outliers.sum()

outlier_counts = pd.DataFrame.from_dict(
    outlier_counts,
    orient='index',
    columns=['Extreme Outlier Count']
)

outlier_counts

,Extreme Outlier Count
Price Per Unit,0
Quantity,0
Total Spent,0


So, the two remaining problems we've identified are the missing values and the dates being in the wrong format. Let's fix the dates first!

In [39]:
# Convert Transaction Date values to datetime
sales_df['Transaction Date'] = pd.to_datetime(sales_df['Transaction Date'], format='%Y-%m-%d')

sales_df['Transaction Date'].head()

,Transaction Date
0,2023-09-08
1,2023-05-16
2,2023-07-19
3,2023-04-27
4,2023-06-11


To handle null values, we can either drop rows with null values or impute data. Imputing means that we use logical techniques to fill in the missing values.

In [40]:
# Dropping all rows with nulls
sales_df_dropped = sales_df.dropna()
sales_df_dropped.shape

(3089, 8)

The problem with dropping all rows with nulls is that we've gotten rid of more than half of our data.

In [41]:
# Show rows where Item is null
sales_df[sales_df['Item'].isnull()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
6,TXN_4433211,NaN,3.0,3.0,9.0,NaN,Takeaway,2023-10-06
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
14,TXN_8915701,NaN,2.0,1.5,3.0,NaN,In-store,2023-03-21
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
31,TXN_8927252,NaN,2.0,1.0,NaN,Credit Card,NaN,2023-11-06
...,...,...,...,...,...,...,...,...
9951,TXN_4122925,NaN,4.0,1.0,4.0,NaN,Takeaway,2023-10-20
9958,TXN_4125474,NaN,2.0,5.0,10.0,Credit Card,In-store,2023-08-02
9981,TXN_4583012,NaN,5.0,4.0,20.0,Digital Wallet,NaN,2023-02-27
9994,TXN_7851634,NaN,4.0,4.0,16.0,NaN,NaN,2023-01-08


Let's first try dropping rows with 3 or more nulls, meaning that at least 6 of the columns have to have valid data.

In [42]:
# Drop rows with 2 or more nulls
sales_df_cleaned = sales_df.dropna(thresh=6)
sales_df_cleaned.shape

(9510, 8)

In [43]:
# Show null counts
null_counts = sales_df_cleaned.isnull().sum()
print(null_counts)

Transaction ID         0
Item                 750
Quantity             358
Price Per Unit       406
Total Spent          390
Payment Method      2778
Location            3541
Transaction Date     334
dtype: int64


Now, let's try imputing data. For categorical variables, the best method is to simply impute "Unknown" for all null values.

In [44]:
# Impute "Unknown" for nulls in Item, Payment Method, and Location
sales_df_cleaned['Item'] = sales_df_cleaned['Item'].fillna('Unknown')
sales_df_cleaned['Payment Method'] = sales_df_cleaned['Payment Method'].fillna('Unknown')
sales_df_cleaned['Location'] = sales_df_cleaned['Location'].fillna('Unknown')

/tmp/ipykernel_1440/3903670014.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_df_cleaned['Item'] = sales_df_cleaned['Item'].fillna('Unknown')
/tmp/ipykernel_1440/3903670014.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_df_cleaned['Payment Method'] = sales_df_cleaned['Payment Method'].fillna('Unknown')
/tmp/ipykernel_1440/3903670014.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



Now, usually we would impute numeric values with the mean or median. However, we know that the relationship between quantity, price per unit, and total spent is quantity * price per unit = total spent, we can fill these values logically.

In [45]:
# Fill missing values in Quantity, Price Per Unit, and Total Spent using q * ppu = ts
sales_df_cleaned['Quantity'] = sales_df_cleaned['Quantity'].fillna(sales_df_cleaned['Total Spent'] / sales_df_cleaned['Price Per Unit'])
sales_df_cleaned['Price Per Unit'] = sales_df_cleaned['Price Per Unit'].fillna(sales_df_cleaned['Total Spent'] / sales_df_cleaned['Quantity'])
sales_df_cleaned['Total Spent'] = sales_df_cleaned['Quantity'] * sales_df_cleaned['Price Per Unit']

/tmp/ipykernel_1440/2910328278.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_df_cleaned['Quantity'] = sales_df_cleaned['Quantity'].fillna(sales_df_cleaned['Total Spent'] / sales_df_cleaned['Price Per Unit'])
/tmp/ipykernel_1440/2910328278.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_df_cleaned['Price Per Unit'] = sales_df_cleaned['Price Per Unit'].fillna(sales_df_cleaned['Total Spent'] / sales_df_cleaned['Quantity'])
/tmp/ipykernel_1440/2910328278.py:4: SettingWithCopyWarning: 
A va

In [46]:
# Show null counts
null_counts = sales_df_cleaned.isnull().sum()
print(null_counts)

Transaction ID        0
Item                  0
Quantity             14
Price Per Unit       14
Total Spent          24
Payment Method        0
Location              0
Transaction Date    334
dtype: int64


We are now left with rows where are numeric values are impossible to be calculated, so we will drop these. We will leave the null values in Transaction Date, since they are much more difficult to impute.

In [47]:
sales_df_cleaned = sales_df_cleaned.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent'])

Now let's look at our new, cleaned dataset.

In [48]:
sales_df_cleaned.shape

(9486, 8)

In [49]:
# Show data types of each column
sales_df_cleaned.dtypes

,0
Transaction ID,object
Item,object
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


In [50]:
# Show summary statistics for the numerical data
sales_df_cleaned.describe(include="number")

,Quantity,Price Per Unit,Total Spent
count,9486.000000,9486.000000,9486.000000
mean,3.030150,2.945657,8.943654
std,1.420068,1.279259,6.012609
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000
